# model 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
API_key = os.getenv("GOOGLE_API_KEY")
llm_model = "gemini-2.5-flash"

In [4]:
os.environ["LANGCHAIN_PROJECT"] = "memory schema collection"

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
model = ChatGoogleGenerativeAI(
                    model=llm_model,
                    temperature=0,
                    timeout=None,
                    max_retries=2)

# Goals 

- Sometimes we want to save memories to a collection rather than single profile.
- We'll also show how to use Trustcall to update this collection.    

# Defining a collection schema
- We can define a collection schema as a Pydantic object.

In [7]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="A list of memories about the user.")

In [8]:
from langchain_core.messages import HumanMessage

In [9]:
# Bind schema to model
model_with_structure = model.with_structured_output(MemoryCollection)

In [12]:
# Invoke the model to produce structured output that matches the schema
memory_collection = model_with_structure.invoke([HumanMessage("My name is Lance. I like to bike.")])

In [13]:
memory_collection.memories

[Memory(content="User's name is Lance."),
 Memory(content='User likes to bike.')]

In [15]:
memory_collection.memories[0].model_dump()

{'content': "User's name is Lance."}

In [16]:
memory_collection.memories[1].model_dump()

{'content': 'User likes to bike.'}

## Save dictionary representation of each memory to the store.

In [17]:
import uuid
from langgraph.store.memory import InMemoryStore

In [18]:
# Initialize the in-memory store
in_memory_store = InMemoryStore()

In [19]:
# Namespace for the memory to save
user_id = "1"
namespace_for_memory = (user_id, "memories")

In [21]:
key = str(uuid.uuid4())
value = memory_collection.memories[0].model_dump()

In [22]:
in_memory_store.put(namespace_for_memory, key, value)

In [26]:
key = str(uuid.uuid4())
value = memory_collection.memories[1].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

## Search for memories in the store.

In [27]:
# Search 
for m in in_memory_store.search(namespace_for_memory):
    print(m.dict())

{'namespace': ['1', 'memories'], 'key': '435bea15-34a3-4f4e-b5f5-db51c514fff6', 'value': {'content': "User's name is Lance."}, 'created_at': '2026-04-19T15:39:43.639319+00:00', 'updated_at': '2026-04-19T15:39:43.639322+00:00', 'score': None}
{'namespace': ['1', 'memories'], 'key': 'e14dc3f6-67f2-489b-8c4c-40cd98e56407', 'value': {'content': 'User likes to bike.'}, 'created_at': '2026-04-19T15:40:04.005848+00:00', 'updated_at': '2026-04-19T15:40:04.005852+00:00', 'score': None}


# Updating collection schema

- Now we'll show that Trustcall can be also used to update a collection.
- As before, we provide the schema for each memory, Memory.
- But, we can supply enable_inserts=True to allow the extractor to insert new memories to the collection.

In [98]:
from trustcall import create_extractor

# Create the extractor
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True, # Update old memory  Add new memory
)

In [99]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Instruction
instruction = """Extract memories and info like name from the following conversation:"""
# Conversation
conversation = [HumanMessage(content="Hi, I'm Lance."), 
                AIMessage(content="Nice to meet you, Lance."), 
                HumanMessage(content="This morning I had a nice bike ride in San Francisco.")]

In [100]:
# Invoke the extractor
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=instruction)] + conversation})

In [101]:
# Messages contain the tool calls
for m in result["messages"]:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (de8a3a5e-365c-40db-a44e-0bb2ccece700)
 Call ID: de8a3a5e-365c-40db-a44e-0bb2ccece700
  Args:
    content: User's name is Lance.
  Memory (e566e03a-4c93-4de9-b133-ce76b53c0b9e)
 Call ID: e566e03a-4c93-4de9-b133-ce76b53c0b9e
  Args:
    content: User had a bike ride in San Francisco this morning.


In [102]:
# Responses contain the memories that adhere to the schema
for m in result["responses"]: 
    print(m)

content="User's name is Lance."
content='User had a bike ride in San Francisco this morning.'


In [103]:
# Metadata contains the tool call  
for m in result["response_metadata"]: 
    print(m)

{'id': 'de8a3a5e-365c-40db-a44e-0bb2ccece700'}
{'id': 'e566e03a-4c93-4de9-b133-ce76b53c0b9e'}


In [104]:
# Update the conversation
updated_conversation = [AIMessage(content="That's great, did you do after?"), 
                        HumanMessage(content="I went to Tartine and ate a croissant."),                        
                        AIMessage(content="What else is on your mind?"),
                        HumanMessage(content="I was thinking about my Japan, and going back this winter!"),]

In [105]:
# Update the instruction
system_msg = """Update existing memories and create new ones based on the following conversation:"""

In [106]:
# We'll save existing memories, giving them an ID, key (tool name), and value
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories

[('0', 'Memory', {'content': "User's name is Lance."}),
 ('1',
  'Memory',
  {'content': 'User had a bike ride in San Francisco this morning.'})]

In [107]:
# Invoke the extractor with our updated conversation and existing memories
result = trustcall_extractor.invoke({"messages": updated_conversation, 
                                     "existing": existing_memories})

In [108]:
# Messages from the model indicate two tool calls were made
for m in result["messages"]:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (ce32be3a-e957-4ca2-9cfa-6d0db7a335d6)
 Call ID: ce32be3a-e957-4ca2-9cfa-6d0db7a335d6
  Args:
    content: User went to Tartine and ate a croissant.
  Memory (60f99df5-c6b8-4237-8432-65cff6d1516b)
 Call ID: 60f99df5-c6b8-4237-8432-65cff6d1516b
  Args:
    content: User is thinking about Japan and going back this winter.


In [109]:
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories

[('0', 'Memory', {'content': 'User went to Tartine and ate a croissant.'}),
 ('1',
  'Memory',
  {'content': 'User is thinking about Japan and going back this winter.'})]

In [110]:
# Responses contain the memories that adhere to the schema
for m in result["responses"]: 
    print(m)

content='User went to Tartine and ate a croissant.'
content='User is thinking about Japan and going back this winter.'


In [111]:
for r in results:
    print(r)

Item(namespace=['1', 'memories'], key='435bea15-34a3-4f4e-b5f5-db51c514fff6', value={'content': "User's name is Lance."}, created_at='2026-04-19T15:39:43.639319+00:00', updated_at='2026-04-19T15:39:43.639322+00:00', score=None)
Item(namespace=['1', 'memories'], key='e14dc3f6-67f2-489b-8c4c-40cd98e56407', value={'content': 'User likes to bike.'}, created_at='2026-04-19T15:40:04.005848+00:00', updated_at='2026-04-19T15:40:04.005852+00:00', score=None)


In [112]:
# Metadata contains the tool call  
for m in result["response_metadata"]: 
    print(m)

{'id': 'ce32be3a-e957-4ca2-9cfa-6d0db7a335d6'}
{'id': '60f99df5-c6b8-4237-8432-65cff6d1516b'}


In [113]:
for m in result["responses"]: 
    print(m)

content='User went to Tartine and ate a croissant.'
content='User is thinking about Japan and going back this winter.'
